# SDK-Sovereign Hardened Training Pipeline

All-in-one notebook with **aggressive logging and auto-backup** so that disconnects don't lose work.

## What this notebook does differently from the previous one

1. **Localhost env server** — no WebSocket timeouts to remote HF Space during training
2. **HF Hub auto-backup every N steps** — even if Colab disconnects, latest checkpoint is on Hub
3. **Verbose JSONL logs** — every action, every reward, every step saved locally to disk
4. **W&B with system metrics** — GPU memory, CPU, disk usage tracked
5. **Resume support** — re-running this notebook continues from latest HF Hub checkpoint
6. **Eval baseline before training** — establishes baseline metrics in W&B

## Required Colab Secrets (Tools menu → Secrets)
- `HF_TOKEN` (write-scoped, must have access to gated Llama models if using them)
- `WANDB_API_KEY`

## Runtime
- GPU: T4 (free tier OK, A10G better if available)
- Run cells top to bottom
- Total time: ~2.5-3 hours on T4

## What gets saved permanently (survives disconnect)
- W&B run at `https://wandb.ai/<your-entity>/sdk-sovereign-hardened`
- HF Hub repo at `https://huggingface.co/<HF_USER>/sdk-sovereign-checkpoints` — full checkpoints every save_steps
- HF Hub repo at `https://huggingface.co/<HF_USER>/sdk-sovereign-lead-adapter` — final Lead adapter
- HF Hub repo at `https://huggingface.co/<HF_USER>/sdk-sovereign-auditor-adapter` — final Auditor adapter


## Cell 1 — Install dependencies

In [ ]:
# Cell 1 — Pinned dependency stack
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q -U "trl==0.24.0" "datasets==3.6.0" "accelerate==1.6.0" "pydantic==2.10.6" peft bitsandbytes
!pip install -q "openenv-core[core] @ git+https://github.com/meta-pytorch/OpenEnv@v0.2.3"
!pip install -q wandb huggingface_hub matplotlib numpy psutil

print('OK: dependencies installed')

## Cell 2 — Configuration and authentication

In [ ]:
# Cell 2 — Configuration, secrets, repo clone
import os
import sys
import json
import warnings
import subprocess
import time
from pathlib import Path
from datetime import datetime

from google.colab import userdata
from huggingface_hub import login, HfApi, create_repo

warnings.filterwarnings('ignore', message='Both `max_new_tokens`')
warnings.filterwarnings('ignore', category=FutureWarning, module='transformers.modeling_attn_mask_utils')

# === EDIT THESE ===
GH_REPO = 'ishansurdi/SDK-Sovereign'
HF_USER = 'ishansurdi'
WANDB_PROJECT = 'sdk-sovereign-hardened'
WANDB_ENTITY = ''  # empty = use default account; do NOT put API key name here

# === HF Hub backup repos (created automatically) ===
CHECKPOINT_REPO = f'{HF_USER}/sdk-sovereign-checkpoints'
LEAD_ADAPTER_REPO = f'{HF_USER}/sdk-sovereign-lead-adapter'
AUDITOR_ADAPTER_REPO = f'{HF_USER}/sdk-sovereign-auditor-adapter'

# === Training scale ===
N_ROLLOUT_EPISODES = 80
N_EVAL_EPISODES = 30
SAVE_EVERY_N_STEPS = 25  # checkpoint to HF Hub every 25 training steps
LOG_EVERY_N_STEPS = 2

# === Phase toggles ===
RUN_SMOKE = True
RUN_BASELINE_EVAL = True  # eval BEFORE training to establish baseline
RUN_ROLLOUTS = True
RUN_TRAIN_LEAD = True
RUN_TRAIN_AUDITOR = True
RUN_TRAINED_EVAL = True
RUN_PLOTS = True
PUSH_FINAL_ADAPTERS = True

# === Behavior flags ===
FORCE_ROLLOUT_REBUILD = False
FORCE_TRAIN_REBUILD = False
RESUME_FROM_HF_HUB = True  # if checkpoints exist on Hub, resume from there

# === Paths ===
BASE_DIR = Path('/content')
LOGS_DIR = BASE_DIR / 'logs'
CHECKPOINTS_DIR = BASE_DIR / 'checkpoints'
ARTIFACTS_DIR = BASE_DIR / 'artifacts'
PLOTS_DIR = BASE_DIR / 'plots'

for d in [LOGS_DIR, CHECKPOINTS_DIR, ARTIFACTS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
print(f'RUN_ID: {RUN_ID}')

# === Authenticate ===
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
os.environ['HF_TOKEN'] = hf_token

wandb_key = userdata.get('WANDB_API_KEY')
if wandb_key:
    os.environ['WANDB_API_KEY'] = wandb_key
    import wandb
    wandb.login(key=wandb_key)
    print('OK: W&B authenticated')

# === Clone repo ===
if not Path('sdk_sovereign_pkg').exists():
    subprocess.run(['git', 'clone', f'https://github.com/{GH_REPO}.git', 'sdk_sovereign_pkg'], check=True)
else:
    subprocess.run(['git', 'pull'], cwd='sdk_sovereign_pkg', check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', 'sdk_sovereign_pkg/'], check=True)
sys.path.insert(0, 'sdk_sovereign_pkg')

# === Create HF Hub backup repos ===
api = HfApi()
for repo in [CHECKPOINT_REPO, LEAD_ADAPTER_REPO, AUDITOR_ADAPTER_REPO]:
    try:
        create_repo(repo, repo_type='model', private=False, exist_ok=True)
        print(f'OK: HF repo ready: {repo}')
    except Exception as e:
        print(f'WARN: Could not create {repo}: {e}')

# === Persistent run state file ===
RUN_STATE_PATH = LOGS_DIR / f'run_state_{RUN_ID}.json'
run_state = {
    'run_id': RUN_ID,
    'started_at': datetime.now().isoformat(),
    'gh_repo': GH_REPO,
    'hf_user': HF_USER,
    'wandb_project': WANDB_PROJECT,
    'phases_completed': [],
}
RUN_STATE_PATH.write_text(json.dumps(run_state, indent=2))

print(f'\nOK: Cell 2 complete. Run state: {RUN_STATE_PATH}')

## Cell 3 — Start localhost env server (avoids WebSocket timeouts)

In [ ]:
# Cell 3 — Localhost env server
import requests
import subprocess
import time
import signal

# Kill any previous uvicorn
subprocess.run(['pkill', '-f', 'uvicorn'], check=False)
time.sleep(2)

# Try common module paths (the right one depends on your repo layout)
candidate_modules = [
    'server.app:app',
    'sdk_sovereign.server.app:app',
    'openenv_sdk_sovereign.server.app:app',
]

env_proc = None
working_module = None
for module_path in candidate_modules:
    try:
        proc = subprocess.Popen(
            ['uvicorn', module_path, '--host', '0.0.0.0', '--port', '8000'],
            cwd='sdk_sovereign_pkg',
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
        )
        time.sleep(8)
        # Check if it's responding
        r = requests.get('http://localhost:8000/health', timeout=5)
        if r.status_code == 200:
            env_proc = proc
            working_module = module_path
            print(f'OK: env server running with module={module_path}')
            print(f'    Health: {r.json()}')
            break
        else:
            proc.terminate()
            time.sleep(1)
    except Exception as e:
        try:
            proc.terminate()
            time.sleep(1)
        except Exception:
            pass

if env_proc is None:
    print('WARN: Could not start localhost env server with any candidate module path.')
    print('      Falling back to remote HF Space (training will be slower and may timeout).')
    ENV_URL = 'https://ishansurdi-sdk-sovereign.hf.space'
else:
    ENV_URL = 'http://localhost:8000'

print(f'\nENV_URL = {ENV_URL}')

# === Logging helper that survives disconnect ===
class StepLogger:
    """Append-only JSONL logger. Flushes after every write."""
    def __init__(self, name):
        self.path = LOGS_DIR / f'{name}_{RUN_ID}.jsonl'
        self.path.touch()
        self._fp = open(self.path, 'a', buffering=1)  # line-buffered

    def log(self, **kwargs):
        kwargs['_ts'] = datetime.now().isoformat()
        self._fp.write(json.dumps(kwargs) + '\n')
        self._fp.flush()

    def close(self):
        self._fp.close()

step_logger = StepLogger('step_log')
print(f'OK: step logger writing to {step_logger.path}')

## Cell 4 — Load model, agents, and HF Hub helpers

In [ ]:
# Cell 4 — Load model + agents + HF Hub backup helpers
from collections import defaultdict
from server.llm_agents import load_model_with_two_adapters, make_agent_pair
from client import SDKSovereignEnv
from models import SDKAction, SDKObservation


def load_grpo_symbols():
    try:
        from trl.trainer.grpo_trainer import GRPOTrainer, GRPOConfig
        import trl
        print(f'Using trl=={trl.__version__}')
        return GRPOTrainer, GRPOConfig
    except Exception:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', '-U',
            'trl==0.24.0', 'datasets==3.6.0', 'accelerate==1.6.0', 'pydantic==2.10.6'
        ])
        for key in list(sys.modules.keys()):
            if key == 'trl' or key.startswith('trl.'):
                del sys.modules[key]
        from trl.trainer.grpo_trainer import GRPOTrainer, GRPOConfig
        import trl
        print(f'Using trl=={trl.__version__}')
        return GRPOTrainer, GRPOConfig


def init_wandb_run(run_name, config=None):
    """Initialize W&B with safe entity fallback."""
    import wandb
    kwargs = {
        'project': WANDB_PROJECT,
        'name': f'{run_name}_{RUN_ID}',
        'config': config or {},
        'reinit': True,
    }
    if WANDB_ENTITY:
        kwargs['entity'] = WANDB_ENTITY
    try:
        return wandb.init(**kwargs)
    except Exception as exc:
        print(f'W&B init with entity failed: {exc}')
        kwargs.pop('entity', None)
        return wandb.init(**kwargs)


def push_to_hub_safe(folder_path, repo_id, commit_msg, path_in_repo=None):
    """Push folder to HF Hub. Never raises - logs failures and continues."""
    try:
        upload_kwargs = {
            'folder_path': str(folder_path),
            'repo_id': repo_id,
            'repo_type': 'model',
            'commit_message': commit_msg,
        }
        if path_in_repo:
            upload_kwargs['path_in_repo'] = path_in_repo
        api.upload_folder(**upload_kwargs)
        print(f'  pushed to https://huggingface.co/{repo_id}')
        return True
    except Exception as e:
        print(f'  WARN: HF push failed for {repo_id}: {e}')
        step_logger.log(event='hf_push_fail', repo=repo_id, error=str(e))
        return False


def load_jsonl(path):
    p = Path(path)
    if not p.exists():
        return []
    rows = [ln for ln in p.read_text().splitlines() if ln.strip()]
    return [json.loads(r) for r in rows]


def save_jsonl(path, rows):
    Path(path).write_text('\n'.join(json.dumps(r) for r in rows))


# === GPU memory check ===
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}, '
          f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# === Load model ===
print('\nLoading base model + LoRA adapters...')
model, tokenizer = load_model_with_two_adapters()
agents = make_agent_pair(model, tokenizer)
print(f'OK: model loaded, VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

step_logger.log(event='model_loaded', vram_gb=torch.cuda.memory_allocated() / 1e9)

## Cell 5 — Smoke test (verify env + model + adapters work)

In [ ]:
# Cell 5 — Smoke test
if RUN_SMOKE:
    print(f'Connecting to: {ENV_URL}')
    smoke_rewards = []
    for ep_idx in range(3):  # 3 smoke episodes
        try:
            with SDKSovereignEnv(base_url=ENV_URL).sync() as env:
                obs = env.reset()
                ep_log = []
                for turn in range(7):
                    agent = agents[obs.current_role]
                    action = agent(obs)
                    ep_log.append({
                        'turn': turn,
                        'role': obs.current_role,
                        'action_type': action.action_type,
                    })
                    obs = env.step(action)
                    if obs.done:
                        smoke_rewards.append(float(obs.reward))
                        print(f'Smoke ep {ep_idx}: reward={obs.reward:.2f} | turns={turn+1}')
                        step_logger.log(
                            event='smoke_episode',
                            ep=ep_idx,
                            reward=float(obs.reward),
                            turns=turn + 1,
                            transcript=ep_log,
                        )
                        break
                else:
                    smoke_rewards.append(float(obs.reward))
                    print(f'Smoke ep {ep_idx}: max turns reached, reward={obs.reward:.2f}')
        except Exception as e:
            print(f'Smoke ep {ep_idx} FAILED: {e}')
            step_logger.log(event='smoke_failure', ep=ep_idx, error=str(e))

    print(f'\nSmoke summary: rewards={smoke_rewards}')
    print(f'  mean={sum(smoke_rewards)/len(smoke_rewards):.2f}' if smoke_rewards else '  no successful episodes')

    # Adapter swap check
    dummy_aud = SDKObservation(
        current_role='auditor', turn_index=0, max_turns=7, error_log='test',
        conversation_history=[], visible_allowlist=['razorpay'], visible_codebase=None,
        visible_filename=None, current_proposal=None, approved_replacement=None,
        done=False, reward=0.0, reward_breakdown={},
    )
    agents['auditor'](dummy_aud)
    assert model.active_adapter == 'auditor_adapter'

    dummy_lead = SDKObservation(
        current_role='lead', turn_index=1, max_turns=7, error_log='test',
        conversation_history=[], visible_allowlist=None, visible_codebase='import stripe',
        visible_filename='payment.py', current_proposal=None, approved_replacement=None,
        done=False, reward=0.0, reward_breakdown={},
    )
    agents['lead'](dummy_lead)
    assert model.active_adapter == 'lead_adapter'
    print('OK: smoke and adapter swap checks passed')
else:
    print('Skipped smoke test by config')

## Cell 6 — Baseline evaluation (untrained model)

In [ ]:
# Cell 6 — Baseline eval BEFORE training
# This gives us the 'before' numbers for the comparison plot
if RUN_BASELINE_EVAL:
    baseline_results = []
    print(f'Running {N_EVAL_EPISODES} baseline episodes...')

    for ep in range(N_EVAL_EPISODES):
        try:
            with SDKSovereignEnv(base_url=ENV_URL).sync() as env:
                obs = env.reset()
                total_reward = 0.0
                transcript = []
                while not obs.done and obs.turn_index < 7:
                    action = agents[obs.current_role](obs)
                    transcript.append({
                        'turn': obs.turn_index,
                        'role': obs.current_role,
                        'action_type': action.action_type,
                    })
                    obs = env.step(action)
                    total_reward += float(obs.reward)
                state = env.state()
                tr = state.test_results or {}
                result = {
                    'phase': 'baseline',
                    'episode': ep,
                    'total_reward': total_reward,
                    'tests_passed': sum(tr.values()) if tr else 0,
                    'tests_total': len(tr) or 3,
                    'success': bool(tr and all(tr.values())),
                    'turns': state.step_count,
                    'repo': state.repo_id,
                    'terminated': state.terminated_reason,
                    'transcript': transcript,
                }
                baseline_results.append(result)
                step_logger.log(event='baseline_eval', **result)
            if ep % 5 == 0:
                pass_rate = sum(r['success'] for r in baseline_results) / len(baseline_results)
                print(f'  ep {ep}: pass_rate_so_far={pass_rate:.2f}')
        except Exception as e:
            print(f'  ep {ep} FAILED: {e}')
            step_logger.log(event='baseline_eval_fail', episode=ep, error=str(e))

    save_jsonl(LOGS_DIR / f'baseline_results_{RUN_ID}.jsonl', baseline_results)

    if baseline_results:
        baseline_pass_rate = sum(r['success'] for r in baseline_results) / len(baseline_results)
        baseline_mean_reward = sum(r['total_reward'] for r in baseline_results) / len(baseline_results)
        print(f'\nBaseline pass rate: {baseline_pass_rate:.2%}')
        print(f'Baseline mean reward: {baseline_mean_reward:.3f}')

        # Save to HF Hub immediately as proof
        push_to_hub_safe(
            LOGS_DIR,
            CHECKPOINT_REPO,
            f'baseline eval results {RUN_ID}',
            path_in_repo=f'logs_{RUN_ID}',
        )
    else:
        print('No baseline results - check env connection')
else:
    print('Skipped baseline eval by config')
    baseline_results = []

## Cell 7 — Rollout collection (resume-safe)

In [ ]:
# Cell 7 — Collect rollouts with resume support
if RUN_ROLLOUTS:
    rollout_lead_path = LOGS_DIR / f'rollout_lead_{RUN_ID}.jsonl'
    rollout_auditor_path = LOGS_DIR / f'rollout_auditor_{RUN_ID}.jsonl'
    rollout_data = {
        'lead': load_jsonl(rollout_lead_path),
        'auditor': load_jsonl(rollout_auditor_path),
    }

    if rollout_data['lead'] and rollout_data['auditor'] and not FORCE_ROLLOUT_REBUILD:
        done_lead = max((r.get('episode', -1) for r in rollout_data['lead']), default=-1) + 1
        done_aud = max((r.get('episode', -1) for r in rollout_data['auditor']), default=-1) + 1
        start_ep = min(done_lead, done_aud)
        print(f'Resuming rollouts from episode {start_ep}')
    else:
        if FORCE_ROLLOUT_REBUILD:
            rollout_data = {'lead': [], 'auditor': []}
        start_ep = 0

    for ep in range(start_ep, N_ROLLOUT_EPISODES):
        try:
            with SDKSovereignEnv(base_url=ENV_URL).sync() as env:
                obs = env.reset()
                per_role = []
                while not obs.done and obs.turn_index < 7:
                    agent = agents[obs.current_role]
                    agent.model.set_adapter(agent.adapter_name)
                    prompt = agent._build_prompt(obs)
                    completion = agent._generate(prompt)
                    action = agent._parse_action(completion)
                    new_obs = env.step(action)
                    per_role.append({
                        'episode': ep,
                        'role': obs.current_role,
                        'prompt': prompt,
                        'completion': completion,
                        'action_type': action.action_type,
                        'reward': float(new_obs.reward),
                    })
                    obs = new_obs

            for r in per_role:
                rollout_data[r['role']].append({
                    'episode': r['episode'],
                    'prompt': r['prompt'],
                    'completion': r.get('completion', ''),
                    'action_type': r['action_type'],
                    'reward': r['reward'],
                })

            # Save after every episode (so disconnect doesn't lose data)
            save_jsonl(rollout_lead_path, rollout_data['lead'])
            save_jsonl(rollout_auditor_path, rollout_data['auditor'])

            if ep % 5 == 0:
                print(f'  rollout {ep}/{N_ROLLOUT_EPISODES}')
                step_logger.log(event='rollout_progress', ep=ep)
        except Exception as e:
            print(f'  rollout ep {ep} FAILED: {e}')
            step_logger.log(event='rollout_fail', ep=ep, error=str(e))

    print(f'\nLead samples: {len(rollout_data["lead"])}')
    print(f'Auditor samples: {len(rollout_data["auditor"])}')

    # Push rollouts to HF Hub
    push_to_hub_safe(
        LOGS_DIR,
        CHECKPOINT_REPO,
        f'rollouts {RUN_ID}',
        path_in_repo=f'logs_{RUN_ID}',
    )
else:
    print('Skipped rollouts by config')
    rollout_data = {'lead': [], 'auditor': []}

## Cell 8 — Train Lead adapter (saves to HF Hub every 25 steps)

In [ ]:
# Cell 8 — Train Lead with aggressive HF Hub backup
if RUN_TRAIN_LEAD and rollout_data['lead']:
    GRPOTrainer, GRPOConfig = load_grpo_symbols()
    from datasets import Dataset
    from transformers import TrainerCallback
    import wandb

    init_wandb_run('lead-grpo-hardened', config={
        'phase': 'lead',
        'n_rollouts': len(rollout_data['lead']),
        'save_every_n_steps': SAVE_EVERY_N_STEPS,
    })

    lead_prompts = [r['prompt'] for r in rollout_data['lead']]
    ds_lead = Dataset.from_dict({'prompt': lead_prompts})

    def lead_reward_fn(completions, **kwargs):
        rewards = []
        for completion in completions:
            try:
                action = agents['lead']._parse_action(completion)
                if action.action_type == 'submit_patch':
                    with SDKSovereignEnv(base_url=ENV_URL).sync() as env:
                        env.reset()
                        env.step(SDKAction(role='auditor', action_type='pass'))
                        env.step(SDKAction(role='lead', action_type='propose_replacement', proposed_sdk='razorpay'))
                        env.step(SDKAction(role='auditor', action_type='approve'))
                        final = env.step(action)
                        rewards.append(float(final.reward))
                elif action.action_type == 'propose_replacement':
                    rewards.append(1.0)
                elif action.action_type == 'request_hint':
                    rewards.append(0.3)
                else:
                    rewards.append(-0.5)
            except Exception as e:
                step_logger.log(event='reward_fn_error', error=str(e))
                rewards.append(-1.0)
        # Log reward distribution to step_logger
        if rewards:
            step_logger.log(
                event='lead_reward_batch',
                mean=sum(rewards) / len(rewards),
                min=min(rewards),
                max=max(rewards),
                n=len(rewards),
            )
        return rewards

    # === Custom callback that pushes checkpoints to HF Hub ===
    class HFHubBackupCallback(TrainerCallback):
        def __init__(self, repo_id, save_every_n=25):
            self.repo_id = repo_id
            self.save_every_n = save_every_n
            self.last_save = -1

        def on_step_end(self, args, state, control, **kwargs):
            if state.global_step > 0 and state.global_step % self.save_every_n == 0:
                if state.global_step != self.last_save:
                    self.last_save = state.global_step
                    print(f'\n[Step {state.global_step}] Backing up to HF Hub...')

                    # Save adapter to local
                    snapshot_dir = ARTIFACTS_DIR / f'lead_step_{state.global_step}'
                    snapshot_dir.mkdir(parents=True, exist_ok=True)
                    try:
                        # Save the lead adapter
                        model.set_adapter('lead_adapter')
                        model.save_pretrained(str(snapshot_dir))
                        # Push to Hub
                        push_to_hub_safe(
                            snapshot_dir,
                            self.repo_id,
                            f'step {state.global_step} backup',
                            path_in_repo=f'step_{state.global_step}',
                        )
                        # Cleanup local copy to save disk
                        import shutil
                        shutil.rmtree(snapshot_dir, ignore_errors=True)
                    except Exception as e:
                        print(f'  Backup failed: {e}')
                        step_logger.log(event='backup_fail', step=state.global_step, error=str(e))

    config = GRPOConfig(
        output_dir=str(CHECKPOINTS_DIR / 'lead'),
        num_generations=4,
        max_completion_length=512,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=5e-6,
        num_train_epochs=1,
        logging_steps=LOG_EVERY_N_STEPS,
        save_steps=SAVE_EVERY_N_STEPS,
        report_to='wandb',
    )

    # Set up adapter freezing
    model.set_adapter('lead_adapter')
    for n, p in model.named_parameters():
        p.requires_grad = ('lead_adapter' in n)

    trainer = GRPOTrainer(
        model=model,
        reward_funcs=lead_reward_fn,
        args=config,
        train_dataset=ds_lead.select(range(min(60, len(ds_lead)))),
        tokenizer=tokenizer,
        callbacks=[HFHubBackupCallback(LEAD_ADAPTER_REPO, SAVE_EVERY_N_STEPS)],
    )

    print(f'Starting Lead training, will save to {LEAD_ADAPTER_REPO} every {SAVE_EVERY_N_STEPS} steps')
    step_logger.log(event='lead_train_start', n_examples=len(ds_lead))

    try:
        trainer.train()
        step_logger.log(event='lead_train_complete')
    except Exception as e:
        print(f'Training crashed: {e}')
        step_logger.log(event='lead_train_crash', error=str(e))
        raise

    # Final save
    final_dir = ARTIFACTS_DIR / 'lead_adapter'
    final_dir.mkdir(parents=True, exist_ok=True)
    model.set_adapter('lead_adapter')
    model.save_pretrained(str(final_dir))
    push_to_hub_safe(final_dir, LEAD_ADAPTER_REPO, 'final lead adapter')

    wandb.finish()
    print('OK: Lead training complete')
else:
    print('Skipped Lead training by config or no rollout data')

## Cell 9 — Train Auditor adapter (saves to HF Hub every 25 steps)

In [ ]:
# Cell 9 — Train Auditor with aggressive HF Hub backup
if RUN_TRAIN_AUDITOR and rollout_data['auditor']:
    GRPOTrainer, GRPOConfig = load_grpo_symbols()
    from datasets import Dataset
    from server.environment import SDKSovereignEnvironment
    from transformers import TrainerCallback
    import wandb

    init_wandb_run('auditor-grpo-hardened', config={
        'phase': 'auditor',
        'n_rollouts': len(rollout_data['auditor']),
    })

    ds_auditor = Dataset.from_dict({'prompt': [x['prompt'] for x in rollout_data['auditor']]})

    def auditor_reward_fn(completions, **kwargs):
        rewards = []
        try:
            allowlist = set(SDKSovereignEnvironment().allowlist)
        except Exception:
            allowlist = set()
        for completion in completions:
            try:
                action = agents['auditor']._parse_action(completion)
                if action.action_type == 'approve':
                    rewards.append(1.0)
                elif action.action_type == 'reject':
                    rewards.append(0.5 if (action.rejection_reason or '').strip() else 0.1)
                elif action.action_type == 'give_hint':
                    text = (action.hint_response or '').lower()
                    hit = any(sdk.lower() in text for sdk in allowlist)
                    rewards.append(0.8 if hit else -0.2)
                elif action.action_type == 'pass':
                    rewards.append(-0.5)
                else:
                    rewards.append(-0.2)
            except Exception as e:
                rewards.append(-1.0)
        if rewards:
            step_logger.log(
                event='auditor_reward_batch',
                mean=sum(rewards) / len(rewards),
                n=len(rewards),
            )
        return rewards

    class AuditorBackupCallback(TrainerCallback):
        def __init__(self, repo_id, save_every_n=25):
            self.repo_id = repo_id
            self.save_every_n = save_every_n
            self.last_save = -1

        def on_step_end(self, args, state, control, **kwargs):
            if state.global_step > 0 and state.global_step % self.save_every_n == 0:
                if state.global_step != self.last_save:
                    self.last_save = state.global_step
                    print(f'\n[Step {state.global_step}] Backing up Auditor...')
                    snapshot_dir = ARTIFACTS_DIR / f'auditor_step_{state.global_step}'
                    snapshot_dir.mkdir(parents=True, exist_ok=True)
                    try:
                        model.set_adapter('auditor_adapter')
                        model.save_pretrained(str(snapshot_dir))
                        push_to_hub_safe(
                            snapshot_dir,
                            self.repo_id,
                            f'step {state.global_step}',
                            path_in_repo=f'step_{state.global_step}',
                        )
                        import shutil
                        shutil.rmtree(snapshot_dir, ignore_errors=True)
                    except Exception as e:
                        step_logger.log(event='auditor_backup_fail', error=str(e))

    config = GRPOConfig(
        output_dir=str(CHECKPOINTS_DIR / 'auditor'),
        num_generations=4,
        max_completion_length=256,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=5e-6,
        num_train_epochs=1,
        logging_steps=LOG_EVERY_N_STEPS,
        save_steps=SAVE_EVERY_N_STEPS,
        report_to='wandb',
    )

    model.set_adapter('auditor_adapter')
    for n, p in model.named_parameters():
        p.requires_grad = ('auditor_adapter' in n)

    trainer = GRPOTrainer(
        model=model,
        reward_funcs=auditor_reward_fn,
        args=config,
        train_dataset=ds_auditor.select(range(min(60, len(ds_auditor)))),
        tokenizer=tokenizer,
        callbacks=[AuditorBackupCallback(AUDITOR_ADAPTER_REPO, SAVE_EVERY_N_STEPS)],
    )

    print(f'Starting Auditor training, will save to {AUDITOR_ADAPTER_REPO} every {SAVE_EVERY_N_STEPS} steps')
    step_logger.log(event='auditor_train_start')

    try:
        trainer.train()
        step_logger.log(event='auditor_train_complete')
    except Exception as e:
        print(f'Training crashed: {e}')
        step_logger.log(event='auditor_train_crash', error=str(e))
        raise

    final_dir = ARTIFACTS_DIR / 'auditor_adapter'
    final_dir.mkdir(parents=True, exist_ok=True)
    model.set_adapter('auditor_adapter')
    model.save_pretrained(str(final_dir))
    push_to_hub_safe(final_dir, AUDITOR_ADAPTER_REPO, 'final auditor adapter')

    wandb.finish()
    print('OK: Auditor training complete')
else:
    print('Skipped Auditor training by config or no rollout data')

## Cell 10 — Trained model evaluation

In [ ]:
# Cell 10 — Eval with trained adapters
if RUN_TRAINED_EVAL:
    import server.llm_agents as la

    # Reload model with trained adapters from Hub
    print('Loading trained adapters from HF Hub...')
    trained_model, trained_tok = la.load_model_with_two_adapters()
    try:
        trained_model.load_adapter(LEAD_ADAPTER_REPO, adapter_name='lead_adapter_trained')
        trained_model.load_adapter(AUDITOR_ADAPTER_REPO, adapter_name='auditor_adapter_trained')
        trained_agents = la.make_agent_pair(trained_model, trained_tok)
        trained_agents['lead'].adapter_name = 'lead_adapter_trained'
        trained_agents['auditor'].adapter_name = 'auditor_adapter_trained'
        print('OK: trained adapters loaded')
    except Exception as e:
        print(f'WARN: Could not load trained adapters from Hub: {e}')
        print('Using current in-memory model (which has trained adapters)')
        trained_agents = agents

    trained_results = []
    print(f'\nRunning {N_EVAL_EPISODES} trained eval episodes...')

    for ep in range(N_EVAL_EPISODES):
        try:
            with SDKSovereignEnv(base_url=ENV_URL).sync() as env:
                obs = env.reset()
                total_reward = 0.0
                transcript = []
                while not obs.done and obs.turn_index < 7:
                    action = trained_agents[obs.current_role](obs)
                    transcript.append({
                        'turn': obs.turn_index,
                        'role': obs.current_role,
                        'action_type': action.action_type,
                    })
                    obs = env.step(action)
                    total_reward += float(obs.reward)
                state = env.state()
                tr = state.test_results or {}
                result = {
                    'phase': 'trained',
                    'episode': ep,
                    'total_reward': total_reward,
                    'tests_passed': sum(tr.values()) if tr else 0,
                    'tests_total': len(tr) or 3,
                    'success': bool(tr and all(tr.values())),
                    'turns': state.step_count,
                    'repo': state.repo_id,
                    'terminated': state.terminated_reason,
                    'transcript': transcript,
                }
                trained_results.append(result)
                step_logger.log(event='trained_eval', **result)
            if ep % 5 == 0:
                pass_rate = sum(r['success'] for r in trained_results) / len(trained_results)
                print(f'  ep {ep}: pass_rate_so_far={pass_rate:.2f}')
        except Exception as e:
            print(f'  ep {ep} FAILED: {e}')

    save_jsonl(LOGS_DIR / f'trained_results_{RUN_ID}.jsonl', trained_results)

    if trained_results:
        trained_pass_rate = sum(r['success'] for r in trained_results) / len(trained_results)
        trained_mean_reward = sum(r['total_reward'] for r in trained_results) / len(trained_results)
        print(f'\nTrained pass rate: {trained_pass_rate:.2%}')
        print(f'Trained mean reward: {trained_mean_reward:.3f}')

        if baseline_results:
            print(f'\nImprovement over baseline:')
            print(f'  Pass rate: {trained_pass_rate - baseline_pass_rate:+.2%}')
            print(f'  Mean reward: {trained_mean_reward - baseline_mean_reward:+.3f}')
else:
    print('Skipped trained eval by config')
    trained_results = []

## Cell 11 — Generate plots

In [ ]:
# Cell 11 — Plots for submission
if RUN_PLOTS and baseline_results and trained_results:
    import matplotlib.pyplot as plt
    import numpy as np
    import statistics

    PLOTS_DIR.mkdir(exist_ok=True)

    # === Plot 1: Pass rate comparison ===
    b_rate = sum(r['success'] for r in baseline_results) / len(baseline_results)
    t_rate = sum(r['success'] for r in trained_results) / len(trained_results)
    plt.figure(figsize=(7, 5))
    bars = plt.bar(['Baseline', 'Trained'], [b_rate, t_rate], color=['#999', '#1f77b4'])
    for bar, val in zip(bars, [b_rate, t_rate]):
        plt.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.1%}',
                 ha='center', fontweight='bold')
    plt.ylabel('Pass rate')
    plt.title(f'Episode pass rate (n={N_EVAL_EPISODES} each)')
    plt.ylim(0, 1.1)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'pass_rate_comparison.png', dpi=150, bbox_inches='tight')
    plt.close()

    # === Plot 2: Mean reward comparison ===
    b_r = statistics.mean(r['total_reward'] for r in baseline_results)
    t_r = statistics.mean(r['total_reward'] for r in trained_results)
    plt.figure(figsize=(7, 5))
    bars = plt.bar(['Baseline', 'Trained'], [b_r, t_r], color=['#999', '#1f77b4'])
    for bar, val in zip(bars, [b_r, t_r]):
        plt.text(bar.get_x() + bar.get_width()/2, val + 0.05, f'{val:+.2f}',
                 ha='center', fontweight='bold')
    plt.ylabel('Mean episode reward')
    plt.axhline(0, color='k', lw=0.5)
    plt.title(f'Mean episode reward (n={N_EVAL_EPISODES} each)')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'mean_reward_comparison.png', dpi=150, bbox_inches='tight')
    plt.close()

    # === Plot 3: Per-episode reward distribution ===
    plt.figure(figsize=(10, 5))
    eps = list(range(N_EVAL_EPISODES))
    b_rewards = [r['total_reward'] for r in baseline_results]
    t_rewards = [r['total_reward'] for r in trained_results]
    plt.plot(eps, b_rewards, 'o-', color='#999', label='Baseline', alpha=0.7)
    plt.plot(eps[:len(t_rewards)], t_rewards, 'o-', color='#1f77b4', label='Trained', alpha=0.7)
    plt.xlabel('Episode')
    plt.ylabel('Total reward')
    plt.title('Per-episode reward, baseline vs trained')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'per_episode_reward.png', dpi=150, bbox_inches='tight')
    plt.close()

    # === Plot 4: Success rate by repo ===
    from collections import defaultdict
    b_by_repo = defaultdict(list)
    t_by_repo = defaultdict(list)
    for r in baseline_results:
        b_by_repo[r['repo']].append(r['success'])
    for r in trained_results:
        t_by_repo[r['repo']].append(r['success'])

    repos = sorted(set(b_by_repo.keys()) | set(t_by_repo.keys()))
    if repos:
        x = np.arange(len(repos))
        b_rates = [sum(b_by_repo.get(r, []))/max(len(b_by_repo.get(r, [])), 1) for r in repos]
        t_rates = [sum(t_by_repo.get(r, []))/max(len(t_by_repo.get(r, [])), 1) for r in repos]
        width = 0.35
        plt.figure(figsize=(10, 5))
        plt.bar(x - width/2, b_rates, width, label='Baseline', color='#999')
        plt.bar(x + width/2, t_rates, width, label='Trained', color='#1f77b4')
        plt.xticks(x, repos, rotation=45, ha='right')
        plt.ylabel('Pass rate')
        plt.title('Pass rate by scenario')
        plt.legend()
        plt.ylim(0, 1.1)
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / 'pass_rate_by_repo.png', dpi=150, bbox_inches='tight')
        plt.close()

    print(f'OK: plots saved to {PLOTS_DIR}/')

    # Push plots to Hub
    push_to_hub_safe(PLOTS_DIR, CHECKPOINT_REPO, f'plots {RUN_ID}', path_in_repo=f'plots_{RUN_ID}')
else:
    print('Skipped plots (need both baseline and trained results)')

## Cell 12 — Final bundle: download everything as one archive

In [ ]:
# Cell 12 — Bundle everything for download
import shutil
from google.colab import files

bundle_name = f'sdk_sovereign_{RUN_ID}'
bundle_dir = BASE_DIR / bundle_name
bundle_dir.mkdir(exist_ok=True)

print('Bundling artifacts...')
for src_name in ['logs', 'checkpoints', 'artifacts', 'plots']:
    src = BASE_DIR / src_name
    if src.exists():
        dst = bundle_dir / src_name
        shutil.copytree(src, dst, dirs_exist_ok=True)
        size_mb = sum(f.stat().st_size for f in dst.rglob('*') if f.is_file()) / 1e6
        print(f'  copied {src_name}: {size_mb:.1f} MB')

# Manifest
manifest_lines = [f'Bundle: {bundle_name}', f'Created: {datetime.now().isoformat()}', '']
for f in sorted(bundle_dir.rglob('*')):
    if f.is_file():
        rel = f.relative_to(bundle_dir)
        size_mb = f.stat().st_size / 1e6
        manifest_lines.append(f'{size_mb:>8.2f} MB  {rel}')
(bundle_dir / 'MANIFEST.txt').write_text('\n'.join(manifest_lines))

# Tar
tar_path = f'/content/{bundle_name}.tar.gz'
print(f'\nCreating archive: {tar_path}')
subprocess.run(['tar', '-czf', tar_path, '-C', str(BASE_DIR), bundle_name], check=True)
size_mb = os.path.getsize(tar_path) / 1e6
print(f'Archive size: {size_mb:.1f} MB')

# Push the entire bundle to HF Hub (one final backup)
push_to_hub_safe(
    bundle_dir,
    CHECKPOINT_REPO,
    f'final bundle {RUN_ID}',
    path_in_repo=f'bundle_{RUN_ID}',
)

# Print where everything is
print('\n' + '='*70)
print('YOUR DATA IS PERMANENTLY STORED AT:')
print('='*70)
print(f'\n1. W&B run: https://wandb.ai/<entity>/{WANDB_PROJECT}')
print(f'\n2. HF Hub backups:')
print(f'   - Lead adapter: https://huggingface.co/{LEAD_ADAPTER_REPO}')
print(f'   - Auditor adapter: https://huggingface.co/{AUDITOR_ADAPTER_REPO}')
print(f'   - Full bundle + logs: https://huggingface.co/{CHECKPOINT_REPO}')
print(f'\n3. Local Colab archive: {tar_path}')

# Trigger browser download (last)
print('\nTriggering browser download...')
try:
    files.download(tar_path)
except Exception as e:
    print(f'Direct download failed: {e}')
    print('Use the HF Hub links above to download instead.')

step_logger.close()
print('\nALL DONE')